# 🤖 CryptoAI: Düşük Parametreli (~1B - 3B) 5 Büyük Dil Modelinin (LLM) Kıyaslama ve Halüsinasyon Benchmark'ı

### T.C. DÜZCE ÜNİVERSİTESİ MÜHENDİSLİK FAKÜLTESİ - BİLGİSAYAR MÜHENDİSLİĞİ BÖLÜMÜ
**BM401 Bilgisayar Mühendisliği Proje Tasarımı**  
* **Ders Sorumlusu:** Prof. Dr. Abdullah Talha KABAKUŞ  
* **Proje Ekibi:** Beyza Ümmühan GÖRYUNAR (231002035) & Ahmet Faruk DURMUŞ (231002013)  
* **Donanım:** Google Colab (Nvidia T4 GPU - 16 GB VRAM)  

---
### 📌 Neden Düşük Parametreli (~1B - 3B) Modelleri Seçtik?
1. **Adil ve Eşit Sıklet Karşılaştırması:** 7B veya 70B'lik dev modeller yerine, birbiriyle doğrudan denk güçteki 1B-3B aralığındaki modeller yarıştırılmıştır.
2. **Mobil ve On-Device Uyumluluğu:** Bu modeller kuantize edildiğinde 1-2 GB RAM'e sığarak cep telefonlarında doğrudan internetsiz çalışabilmektedir.
3. **T4 GPU Hızı:** Bellek taşması (Out-of-Memory) olmadan saniyeler içinde çıkarım ve eğitim (Fine-Tuning/LoRA) imkanı sunar.

### 🎯 Sınanan 5 Model Ailesi:
1. **Llama 3.2 (1.2B)** - Meta AI
2. **DeepSeek-R1-Distill (1.5B)** - DeepSeek AI (En güncel akıl yürütme modeli)
3. **Qwen 2.5 (1.5B)** - Alibaba Cloud (Çok dilli mantık şampiyonu)
4. **Gemma 2 (2.6B)** - Google DeepMind
5. **Ministral / SmolLM2 (1.7B - 3B)** - Mistral AI / Hugging Face


In [ ]:
# 1. ADIM: Gerekli Kütüphanelerin Kurulumu ve İçe Aktarma
!pip install -q transformers torch accelerate bitsandbytes matplotlib seaborn pandas numpy

import os
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Kullanılan Donanım Hızlandırıcı: {device.upper()}")
if device == 'cuda':
    print(f"🚀 GPU Modeli: {torch.cuda.get_device_name(0)}")
    print(f"📦 Toplam VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. ADIM: 50 Soruluk Benchmark Veri Setinin Yüklenmesi (4 Kategori + 10 Tuzak Soru)
BENCHMARK_URL = 'https://raw.githubusercontent.com/beyzagoryunar/CryptoAI-ML/main/crypto_llm_benchmark_50_questions.json'

try:
    with open('crypto_llm_benchmark_50_questions.json', 'r', encoding='utf-8') as f:
        questions = json.load(f)
    print("✅ Yerel benchmark veri seti başarıyla yüklendi.")
except Exception:
    import urllib.request
    urllib.request.urlretrieve(BENCHMARK_URL, 'crypto_llm_benchmark_50_questions.json')
    with open('crypto_llm_benchmark_50_questions.json', 'r', encoding='utf-8') as f:
        questions = json.load(f)
    print("🌐 Benchmark veri seti GitHub üzerinden indirildi.")

df_questions = pd.DataFrame(questions)
print(f"📊 Toplam Soru Sayısı : {len(df_questions)}")
print(f"📂 Kategori Dağılımı  :")
print(df_questions['category'].value_counts())
display(df_questions.head(5))


In [ ]:
# 3. ADIM: TTFT, HIZ VE GECİKME ÖLÇÜM MOTORU (STREAMING PROTOKOLÜ)
def evaluate_model_on_questions(model_name, questions_sample, max_new_tokens=100):
    print(f"\n⏳ Model Yükleniyor: {model_name}...")
    torch.cuda.empty_cache()
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()
    
    ttft_list = []
    latency_list = []
    tokens_per_sec_list = []
    answers = []
    
    # Bellek ölçümü
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    
    print(f"🚀 {len(questions_sample)} Soru Sınanıyor...")
    for idx, item in enumerate(questions_sample):
        q = item['question']
        
        # Prompt hazırlama (Chat formatı)
        prompt = f"<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        generation_kwargs = dict(inputs, streamer=streamer, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=False)
        
        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        
        t0 = time.perf_counter()
        thread.start()
        
        first_token_time = None
        generated_text = ""
        token_count = 0
        
        for text in streamer:
            if first_token_time is None:
                first_token_time = time.perf_counter()
            generated_text += text
            token_count += 1
            
        t_end = time.perf_counter()
        thread.join()
        
        ttft = (first_token_time - t0) * 1000 if first_token_time else 0  # ms
        total_lat = (t_end - t0)  # sn
        tok_sec = token_count / total_lat if total_lat > 0 else 0
        
        ttft_list.append(ttft)
        latency_list.append(total_lat)
        tokens_per_sec_list.append(tok_sec)
        answers.append(generated_text.strip())
        
    avg_ttft = np.mean(ttft_list)
    avg_lat = np.mean(latency_list)
    avg_speed = np.mean(tokens_per_sec_list)
    
    del model
    del tokenizer
    torch.cuda.empty_cache()
    
    return {
        'model_name': model_name,
        'avg_ttft_ms': avg_ttft,
        'avg_latency_s': avg_lat,
        'avg_speed_tok_s': avg_speed,
        'peak_vram_gb': peak_vram,
        'answers': answers
    }


In [ ]:
# 4. ADIM: DÜŞÜK PARAMETRELİ (~1B - 3B) 5 MODELİN T4 GPU'DA YARIŞTIRILMASI
# Birbirine denk, hafif ve mobil uyumlu 5 açık kaynak model:
CANDIDATE_MODELS = [
    "meta-llama/Llama-3.2-1B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "google/gemma-2-2b-it",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct"  # mistralai/Ministral-3B-instruct alternatifi
]

# 50 Soruluk Test (Dengeli alt küme veya tam 50 soru):
test_subset = questions[:15] + questions[40:50] # 15 Teknik/Finans + 10 Tuzak soru

benchmark_results = []
for m in CANDIDATE_MODELS:
    try:
        res = evaluate_model_on_questions(m, test_subset, max_new_tokens=60)
        benchmark_results.append(res)
    except Exception as e:
        print(f"⚠️ {m} çalışırken hata oluştu: {e}")


In [ ]:
# 5. ADIM: HALÜSİNASYON DİRENCİ VE PERFORMANS TABLOSU
# Tuzak sorularda (41-50) modelin 'yoktur / yanlıştır / anonimdir / CEOsu yoktur' deme başarısı:
trap_keywords = ['yoktur', 'değildir', 'yanlış', 'bilinmiyor', 'anonim', 'merkeziyetsiz', 'hayır', 'bağımsız']

summary_rows = []
for res in benchmark_results:
    m_name = res['model_name'].split('/')[-1]
    ans_list = res['answers']
    
    # Tuzak soruların (son 10 soru) incelenmesi
    trap_answers = ans_list[-10:]
    trap_success = sum(1 for a in trap_answers if any(k in a.lower() for k in trap_keywords))
    hallucination_resistance = (trap_success / 10) * 100
    
    # Genel Başarı Skoru (Hız + Doğruluk + Halüsinasyon Direnci sentezi)
    overall_score = (hallucination_resistance * 0.5) + (min(res['avg_speed_tok_s'], 50) * 1.0)
    
    summary_rows.append({
        'Model Adı': m_name,
        'İlk Token Hızı (TTFT ms)': f"{res['avg_ttft_ms']:.1f} ms",
        'Üretim Hızı (Token/sn)': f"{res['avg_speed_tok_s']:.1f} t/s",
        'GPU VRAM (GB)': f"{res['peak_vram_gb']:.2f} GB",
        'Halüsinasyon Direnci': f"%{hallucination_resistance:.1f}",
        'Genel Başarı Puanı (/100)': f"{overall_score:.1f}"
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)


In [ ]:
# 6. ADIM: 4 PANELLİ KARŞILAŞTIRMA GRAFİKLERİ
plt.figure(figsize=(15, 10), facecolor='white')

models_labels = [r['model_name'].split('/')[-1] for r in benchmark_results]
ttft_vals = [r['avg_ttft_ms'] for r in benchmark_results]
speed_vals = [r['avg_speed_tok_s'] for r in benchmark_results]
vram_vals = [r['peak_vram_gb'] for r in benchmark_results]
colors = ['#00B2FE', '#00E676', '#FF9500', '#AF52DE', '#FF3B30']

# 1. Panel: TTFT (Düşük olan daha iyi)
plt.subplot(2, 2, 1)
bars1 = plt.bar(models_labels, ttft_vals, color=colors, width=0.5, edgecolor='black')
plt.title('1. İlk Token Gecikmesi - TTFT (ms - Düşük İyi)', fontweight='bold')
plt.ylabel('Milisaniye (ms)')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars1:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + 5, f"{h:.0f} ms", ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. Panel: Üretim Hızı (Yüksek olan daha iyi)
plt.subplot(2, 2, 2)
bars2 = plt.bar(models_labels, speed_vals, color=colors, width=0.5, edgecolor='black')
plt.title('2. Üretim Akıcılığı (Token / Saniye - Yüksek İyi)', fontweight='bold')
plt.ylabel('Token / Sn')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars2:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + 1, f"{h:.1f} t/s", ha='center', va='bottom', fontsize=9, fontweight='bold')

# 3. Panel: VRAM Tüketimi
plt.subplot(2, 2, 3)
bars3 = plt.bar(models_labels, vram_vals, color=colors, width=0.5, edgecolor='black')
plt.title('3. GPU Bellek Kullanımı - VRAM (GB - Düşük İyi)', fontweight='bold')
plt.ylabel('VRAM (GB)')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars3:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + 0.1, f"{h:.2f} GB", ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Panel: Halüsinasyon Direnci (Tuzak Soru Başarısı)
plt.subplot(2, 2, 4)
trap_scores = [float(r['Halüsinasyon Direnci'].replace('%', '')) for r in summary_rows]
bars4 = plt.bar(models_labels, trap_scores, color=colors, width=0.5, edgecolor='black')
plt.title('4. Tuzak Sorularda Halüsinasyon Direnci (Doğruluk %)', fontweight='bold')
plt.ylabel('Başarı Yüzdesi (%)')
plt.ylim(0, 115)
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars4:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + 2, f"%{h:.0f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('llm_benchmark_4_panel_karsilastirma.png', dpi=300)
plt.show()
print("✅ Benchmark tamamlandı ve grafik 'llm_benchmark_4_panel_karsilastirma.png' olarak kaydedildi!")
